In [2]:
from spark_config import get_spark_session
from pyspark.ml.fpm import FPGrowth

spark = get_spark_session(app_name="GPGrowth")


26/05/08 17:57:02 WARN Utils: Your hostname, khanhdo-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 192.168.118.128 instead (on interface ens33)
26/05/08 17:57:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 17:57:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
df_real_user_portfolio = spark.read.parquet("/home/khanhdo/Documents/project/bigdata_mining/data_processed/real_user_portfolios")
df_real_user_portfolio.show(5, truncate=False)

+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+-----------+
|user_address                              |token_address                             |balance              |tx_count|last_active        |is_contract|
+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+-----------+
|0x00000000fff16d2b960b3ef2686e40ba99403835|0xdac17f958d2ee523a2206206994597c13d831ec7|71.203482            |2       |2026-04-28 01:37:59|false      |
|0x00000000fff16d2b960b3ef2686e40ba99403835|0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2|2.0000001514545227   |8       |2026-04-28 01:54:47|false      |
|0x00006566a3158a643cfa9dbcfa9495a1a00bd7ea|0xdac17f958d2ee523a2206206994597c13d831ec7|0.0010000000000000009|2       |2026-02-08 10:13:59|false      |
|0x0001915c9c3d2911bc91b327e2425a7daa6b1a34|0xdac17f958d2ee523a2206206994597c13d831ec7|2.98155

In [4]:
df_tokens = spark.read.csv("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_tokens.csv", header=True, inferSchema=True)

In [10]:
import pyspark.sql.functions as F

df_portfolio = df_real_user_portfolio.alias("p").join(
    df_tokens.alias("t"), 
    F.col("p.token_address") == F.col("t.address"), 
    "inner"
).select("p.user_address", "t.symbol", "p.balance", "p.tx_count")

In [12]:
df_portfolio.show(5, truncate=False)

+------------------------------------------+------+---------------------+--------+
|user_address                              |symbol|balance              |tx_count|
+------------------------------------------+------+---------------------+--------+
|0x00000000fff16d2b960b3ef2686e40ba99403835|USDT  |71.203482            |2       |
|0x00000000fff16d2b960b3ef2686e40ba99403835|WETH  |2.0000001514545227   |8       |
|0x00006566a3158a643cfa9dbcfa9495a1a00bd7ea|USDT  |0.0010000000000000009|2       |
|0x0001915c9c3d2911bc91b327e2425a7daa6b1a34|USDT  |2.981555974335137E-19|34      |
|0x000191bbf5a21d8a85e402c94b5beed4765580d4|USDT  |22.103434            |1       |
+------------------------------------------+------+---------------------+--------+
only showing top 5 rows



In [ ]:
# Nhóm các token (symbol) theo từng user_address thành một mảng (basket/danh sách items)
df_baskets = df_portfolio.groupBy("user_address").agg(F.collect_set("symbol").alias("items"))
df_baskets.show(5, truncate=False)

+------------------------------------------+---------------------------------------------------+
|user_address                              |items                                              |
+------------------------------------------+---------------------------------------------------+
|0x0000000000000000000000000000000000000001|[ZEON, NOW, KAT, BCDT, WETH, SXUT, MFT, DENT, KYTE]|
|0x00000000000000000000000000000000000012dc|[NMR]                                              |
|0x0000000000000000000000000000000000001cc1|[NMR]                                              |
|0x0000000000000000000000000000000000002c62|[NMR]                                              |
|0x00000000000000000000000000000000000036d6|[NMR]                                              |
+------------------------------------------+---------------------------------------------------+
only showing top 5 rows



In [ ]:
from pyspark.ml.fpm import FPGrowth

# Lọc chỉ lấy những ví cầm từ 2 token trở lên
df_baskets_filtered = df_baskets.filter(F.size(F.col("items")) >= 2)

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.005, minConfidence=0.3)

# Training mô hình trên tập dữ liệu đã lọc
model = fpGrowth.fit(df_baskets_filtered)

# Hiển thị các itemsets phổ biến (Frequent Itemsets)
model.freqItemsets.orderBy("freq", ascending=False).show(30, truncate=False)

+------------+-------+
|items       |freq   |
+------------+-------+
|[USDT]      |3133929|
|[WETH]      |85061  |
|[LINK]      |62950  |
|[WBTC]      |31321  |
|[QNT]       |12036  |
|[WETH, USDT]|7331   |
|[WBTC, USDT]|5891   |
|[BNB]       |5089   |
|[HANDY]     |5022   |
|[LDO]       |5002   |
|[CRO]       |4963   |
|[STORJ]     |4261   |
|[LINK, USDT]|4184   |
|[BAT]       |3766   |
|[DENT]      |3325   |
|[VXT]       |2953   |
|[MANA]      |2890   |
|[WBTC, WETH]|2374   |
|[CHZ]       |2197   |
|[TEL]       |2165   |
|[SNX]       |2091   |
|[ANKR]      |2088   |
|[QNT, LINK] |2055   |
|[NEXO]      |2008   |
|[NKN]       |1869   |
|[TET]       |1860   |
|[FUN]       |1743   |
|[XYO]       |1536   |
|[LEASH]     |1496   |
|[ZRX]       |1456   |
+------------+-------+
only showing top 30 rows



In [37]:
model.associationRules.orderBy("confidence", ascending=False).show(20, truncate=False)

+----------+----------+----------+----+-------+
|antecedent|consequent|confidence|lift|support|
+----------+----------+----------+----+-------+
+----------+----------+----------+----+-------+



In [35]:
# 1. Lưu dưới dạng Parquet (Khuyên dùng trong Spark vì giữ nguyên định dạng Mảng - Array cho 2 cột antecedent, consequent)
output_parquet_path = "/home/khanhdo/Documents/project/bigdata_mining/data_processed/fpgrowth_rules_parquet"
model.associationRules.write.mode("overwrite").parquet(output_parquet_path)

# 2. Lưu dưới dạng CSV (Dễ dàng đọc bằng Excel/Pandas, nhưng phải convert mảng Array thành chuỗi String)
output_csv_path = "/home/khanhdo/Documents/project/bigdata_mining/data_processed/fpgrowth_rules_csv"

df_rules_csv = model.associationRules.withColumn(
    "antecedent", F.concat_ws(",", F.col("antecedent"))
).withColumn(
    "consequent", F.concat_ws(",", F.col("consequent"))
)

# Sắp xếp theo confidence giảm dần trước khi xuất file cho đẹp
df_rules_csv = df_rules_csv.orderBy("confidence", ascending=False)
df_rules_csv.write.mode("overwrite").option("header", "true").csv(output_csv_path)

print(f"Đã lưu thành công vào:\n- {output_parquet_path}\n- {output_csv_path}")

Đã lưu thành công vào:
- /home/khanhdo/Documents/project/bigdata_mining/data_processed/fpgrowth_rules_parquet
- /home/khanhdo/Documents/project/bigdata_mining/data_processed/fpgrowth_rules_csv
